# Adding additional bounding box options for improved Ripley's K assumptions
Replcaing the rectangula get_window() function with a window that acutaly expressed the tissue geometry and then update the edge correction weight in bivariate_k() to work with that new window.

1. Survey options - Spatstat spatial point pattern includes options for: rectangular, polygonal, binary mask, and custom windows.
    - Realistic options are:
        - Convex hull (scipy.spatial.ConvexHull). This is the simplest polygonal upgrade: tight around points but my still include concavitites within the tissue. Good enough if the strip is mostly convex/
        - Alpha shape / concave hull (alphashape package or manual Delaunay-based) - follows the actual tissue boundary including concavities. More accurate but introduces the alpha parameter which controls "tightness". Spatstat's polygonal/mask windows approximate this.
        - Binary mask via kernel density. Estimate a 2D KDE, threshold it, extract the boundary as a polygon. More robsut to noise but computationally heavier and introduces bandwidth + threshold parameters.

        Convex hull is the most obvious starting point.

2. Implement get_window() v2. Refactor to retunr a window object that encodes the window type and geometry. SOme kind of dictionary or small dataclass that can be acted upon by downstream functions.

3. Edge correction for polygonal windows. This is key because the current fraction_inside_rect() parameter exploits rectangular geometry to computs arc fractions analytically. For a gneerla polygon, calculating the area of a circle at point p with radius r that fits inside the shape is more difficult.
    - Approach 1: Monte Carlo edge correction: sample N points uniformly on the circle of radius r and centre p, count the fraction inside the polygon (point-in-polygon test via "shapley" or matplotlib.path). Simple and general but potentially slower due to sampling. Also accuarcy scales as 1/√N.
    - Approach 2: Analytical arc-polygon intersection. Compute the exact arc length inside the polygon by finding intersection point of the circle with each polgyon edge. This is what spatstat does internally. Faster per-query, but substantialy more complex. 

4. Update bivariate_k(). It needs to be able to accept the new window object. The intesnity λ_B = n_B / |W| now requires computing polygonal area (shapely). The edge correction call dispatches to fraction_inside_rect() for rectangular windows or to the new polygonal method.

5. Validation: rectangular vs. polygonal window for the same strip. 

6. Validation: Synthetic CSR on a polygonal window

7. Re-run controls (KRT8 x KRT18, MALAT1 x KRT18, KRT8 x SCGB3A1)

## Overall result:
> Should be that the isotropy assumption is much more resonable given the window is now actually reflective of the tissue shape and area, and the edge correction will also be more accurate. Stationarity is still somewhat violated due to heterogeneity of points within the tissue structure but the permutation envelope somewhat accounts for this by reframing the null. 

